# 06 — Entrenamiento paso a paso de EarthFormer

En esta libreta vamos a reconstruir, en versión docente, el entrenamiento determinístico de **EarthFormer** usado en el pipeline de nowcasting.

La idea no es reemplazar el entrenamiento completo de investigación, sino abrir la caja negra:

1. de dónde sale el script original;
2. cómo se definen los datos y los splits;
3. qué parte del YAML construye la arquitectura;
4. qué forma tiene un batch;
5. cómo corre el `forward` del modelo;
6. cómo se calcula la pérdida;
7. cómo se guarda un checkpoint `.pth`.

El pipeline original del repositorio CasCast entra por:

```bash
bash run_ideam_train.sh
```

Ese script llama internamente:

```bash
python train.py -c configs/sevir_used/EarthFormer_ideam.yaml --outdir experiments/...
```

Aquí hacemos una versión más pequeña y legible, pensada para clase y para ejecutarse también en Hugging Face/Colab con pocos archivos `.npy`.

## 0. Instalación mínima

Primero usa el ambiente base del curso. Para esta libreta se necesitan dos paquetes extra:

```bash
pip install -r training_requirements.txt
```

Si estás en Hugging Face o Colab, es común que `torch` ya esté instalado; en ese caso el comando solo completará lo que falte.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
print("Proyecto:", PROJECT_ROOT)

In [ ]:
import json
import inspect

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from earthformer_training_lab.data import (
    IDEAMSequenceDataset,
    create_train_val_split,
    describe_files,
)
from earthformer_training_lab.model import (
    architecture_summary,
    build_earthformer,
    count_trainable_parameters,
    get_earthformer_params,
    load_yaml,
)
from earthformer_training_lab.train import (
    compute_batch_metrics,
    run_training,
    train_one_epoch,
    validate_one_epoch,
    weighted_mse_loss,
)
from course_utils.palette import rain_cmap, RAIN_LEVELS

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Qué se jaló del repositorio original

Del repositorio original se tomaron las piezas conceptuales que dependen de `run_ideam_train.sh`:

- **Script de entrada original:** `run_ideam_train.sh`.
- **Config original:** `configs/sevir_used/EarthFormer_ideam.yaml`.
- **Dataset original:** `datasets/ideam_used.py`.
- **Arquitectura original:** `networks/earthformer_xy.py`.
- **Wrapper/training original:** `train.py`, `models/non_ar_model.py`, `models/model.py`.

Para el curso dejamos una versión modular en:

```text
earthformer_training_lab/
  architectures/earthformer_xy.py      # arquitectura real EarthFormer_xy copiada del repo CasCast
  config/earthformer_ideam_course.yaml # YAML docente simplificado
  data.py                              # dataset IDEAM-style y splits
  model.py                             # constructor de arquitectura
  train.py                             # loop de entrenamiento explícito
scripts/train_earthformer_course.py    # ejecución por terminal
```

La diferencia importante: el repo original usa wrappers generales, DDP, samplers de casos difíciles y métricas completas. Aquí usamos un loop simple para que se entienda cada paso.

In [ ]:
COURSE_CFG = PROJECT_ROOT / "earthformer_training_lab" / "config" / "earthformer_ideam_course.yaml"
ARCH_FILE = PROJECT_ROOT / "earthformer_training_lab" / "architectures" / "earthformer_xy.py"
DATA_DIR = PROJECT_ROOT / "data" / "samples"
SPLIT_DIR = PROJECT_ROOT / "earthformer_training_lab" / "splits" / "demo"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "training" / "earthformer_course"

print("Config docente:", COURSE_CFG)
print("Arquitectura:", ARCH_FILE)
print("Datos:", DATA_DIR)

## 2. Dataset y convención 13 → 12

Cada archivo `.npy` contiene una secuencia radar con forma:

\[
(T, H, W) = (25, 128, 128)
\]

Usamos la misma convención del resto del curso:

\[
\text{input} = x_{0:13}, \qquad \text{target} = x_{13:25}
\]

Es decir, 13 imágenes observadas para predecir 12 imágenes futuras. Si la resolución temporal es 5 minutos, los horizontes son:

\[
5, 10, 15, \dots, 60 \text{ minutos}
\]

El dataset normaliza la lluvia de `mm/h` a `[0, 1]` dividiendo entre `60.0`. Esto hace más estable el entrenamiento.

In [ ]:
# Si todavia no existen las muestras, ejecuta en terminal:
# python scripts/download_assets.py

split_paths = create_train_val_split(DATA_DIR, SPLIT_DIR, val_fraction=0.2, seed=7)
print(split_paths)

train_files = (SPLIT_DIR / "train.txt").read_text().strip().splitlines()
val_files = (SPLIT_DIR / "val.txt").read_text().strip().splitlines()
print("Train:", train_files)
print("Valid:", val_files)

In [ ]:
metadata = pd.DataFrame(describe_files(DATA_DIR, train_files + val_files))
metadata

In [ ]:
train_ds = IDEAMSequenceDataset(DATA_DIR, SPLIT_DIR / "train.txt", augment=True)
valid_ds = IDEAMSequenceDataset(DATA_DIR, SPLIT_DIR / "val.txt", augment=False)

sample = train_ds[0]
print("Archivo:", sample["file_name"])
print("inputs:", tuple(sample["inputs"].shape), "rango", float(sample["inputs"].min()), float(sample["inputs"].max()))
print("target:", tuple(sample["data_samples"].shape), "rango", float(sample["data_samples"].min()), float(sample["data_samples"].max()))

In [ ]:
cmap, norm = rain_cmap()
inputs_mm = sample["inputs"].squeeze(1).numpy() * 60.0
target_mm = sample["data_samples"].squeeze(1).numpy() * 60.0

fig, axes = plt.subplots(2, 4, figsize=(10, 4.8), constrained_layout=True)
for col, idx in enumerate([0, 4, 8, 12]):
    axes[0, col].imshow(inputs_mm[idx], cmap=cmap, norm=norm)
    axes[0, col].set_title(f"Input t-{(12-idx)*5} min")
    axes[0, col].axis("off")
for col, idx in enumerate([2, 5, 8, 11]):
    im = axes[1, col].imshow(target_mm[idx], cmap=cmap, norm=norm)
    axes[1, col].set_title(f"Target +{(idx+1)*5} min")
    axes[1, col].axis("off")
fig.colorbar(im, ax=axes, ticks=RAIN_LEVELS, shrink=0.8, label="Rain rate (mm/h)")
plt.show()

## 3. Configuración YAML

El YAML docente mantiene el bloque esencial de la configuración original:

```yaml
model:
  type: non_ar_model
  params:
    sub_model:
      EarthFormer_xy:
        in_len: 13
        out_len: 12
        height: 128
        width: 128
```

Ese bloque es el que realmente construye la arquitectura. El resto controla entrenamiento: batch size, learning rate, número de épocas y normalización.

In [ ]:
cfg = load_yaml(COURSE_CFG)
print(json.dumps(cfg["model"]["params"]["sub_model"]["EarthFormer_xy"], indent=2))

## 4. Arquitectura EarthFormer

La clase usada en este curso es la misma `EarthFormer_xy` del repositorio CasCast.

La entrada al modelo tiene forma:

\[
(B, T_{in}, C, H, W) = (B, 13, 1, 128, 128)
\]

La salida tiene forma:

\[
(B, T_{out}, C, H, W) = (B, 12, 1, 128, 128)
\]

Internamente, `EarthFormer_xy` reordena el tensor a `(B, T, H, W, C)`, ejecuta el `CuboidTransformerModel`, y vuelve a la convención de PyTorch.

In [ ]:
from earthformer_training_lab.architectures.earthformer_xy import EarthFormer_xy, earthformer_base

print(inspect.getsource(EarthFormer_xy))

In [ ]:
# Miramos solo la primera parte de earthformer_base para ubicar los hiperparametros principales.
source_lines = inspect.getsource(earthformer_base).splitlines()
print("\n".join(source_lines[:65]))

In [ ]:
model = build_earthformer(cfg)
print(model.__class__.__name__)
print("Parametros entrenables:", f"{count_trainable_parameters(model):,}")
pd.DataFrame(architecture_summary(model))

## 5. Forward pass con un batch

Antes de entrenar, siempre conviene verificar que el modelo acepta el batch y produce la forma esperada.

Este paso responde una pregunta básica: **¿la arquitectura realmente transforma 13 cuadros en 12 cuadros futuros?**

In [ ]:
loader = DataLoader(train_ds, batch_size=1, shuffle=False, num_workers=0)
batch = next(iter(loader))

with torch.no_grad():
    pred = model(batch["inputs"])

print("input:", tuple(batch["inputs"].shape))
print("target:", tuple(batch["data_samples"].shape))
print("prediction:", tuple(pred.shape))
print("loss inicial:", float(weighted_mse_loss(pred, batch["data_samples"])))
print(compute_batch_metrics(pred, batch["data_samples"]))

## 6. Pérdida de entrenamiento

El entrenamiento original usa una variante de MSE ponderada por lluvia. La idea es sencilla:

\[
\mathcal{L} = \frac{1}{N}\sum_i w_i(\hat{y}_i - y_i)^2
\]

con:

\[
w_i = \frac{y_i + 1}{\mathrm{mean}(y + 1)}
\]

Como `y` está normalizado entre 0 y 1, los pixeles con lluvia reciben un poco más de peso que el fondo seco. Esto evita que el modelo aprenda únicamente a predecir valores cercanos a cero.

In [ ]:
print(inspect.getsource(weighted_mse_loss))

## 7. Loop de entrenamiento explícito

El ciclo de entrenamiento contiene los pasos clásicos de PyTorch:

1. pasar `inputs` y `target` al dispositivo;
2. `optimizer.zero_grad()`;
3. `prediction = model(inputs)`;
4. calcular pérdida;
5. `loss.backward()`;
6. `optimizer.step()`;
7. validar y guardar checkpoint.

En el repositorio original esto está repartido entre `train.py`, `models/model.py` y `models/non_ar_model.py`. Aquí lo dejamos concentrado para clase.

In [ ]:
print(inspect.getsource(train_one_epoch))

## 8. Entrenamiento corto de 5 épocas

La siguiente celda entrena EarthFormer con las muestras pequeñas del curso y guarda checkpoints en:

```text
outputs/training/earthformer_course/checkpoints/earthformer/
```

Archivos esperados:

- `checkpoint_latest.pth`
- `checkpoint_best.pth`
- `checkpoint_epoch_001.pth`, ..., `checkpoint_epoch_005.pth`

Para una prueba muy rápida puedes usar `SMOKE_MODE = True`. Para el ejemplo completo de clase, usa `SMOKE_MODE = False`.

In [ ]:
SMOKE_MODE = True   # Cambia a False para correr las 5 epocas completas.
EPOCHS = 5

result = run_training(
    config_path=COURSE_CFG,
    sample_dir=DATA_DIR,
    list_dir=SPLIT_DIR,
    output_dir=OUTPUT_DIR,
    epochs=1 if SMOKE_MODE else EPOCHS,
    batch_size=1 if SMOKE_MODE else 2,
    device="auto",
    seed=7,
    create_splits=True,
    limit_train_batches=1 if SMOKE_MODE else None,
    limit_valid_batches=1 if SMOKE_MODE else None,
)

print("Checkpoint latest:", result["latest_checkpoint"])
print("Checkpoint best:", result["best_checkpoint"])

In [ ]:
history = pd.DataFrame(result["history"])
history

In [ ]:
ckpt = torch.load(result["latest_checkpoint"], map_location="cpu")
print("Llaves principales:", ckpt.keys())
print("Epoca guardada:", ckpt["epoch"])
print("Submodelos:", ckpt["model"].keys())
first_key = next(iter(ckpt["model"]["EarthFormer_xy"].keys()))
print("Primera llave del state_dict:", first_key)

## 9. Lo mismo desde terminal

Para correr el mismo flujo sin notebook:

```bash
python scripts/train_earthformer_course.py --smoke --device auto
```

Para el ejemplo de 5 épocas:

```bash
python scripts/train_earthformer_course.py --epochs 5 --device auto
```

En CPU funciona, pero puede ser lento. En GPU debería ser mucho más cómodo. En Hugging Face/Colab, primero asegúrate de tener los datos:

```bash
python scripts/download_assets.py
pip install -r training_requirements.txt
python scripts/train_earthformer_course.py --epochs 5 --device auto
```

## 10. Preguntas para discusión

1. ¿Por qué normalizamos la lluvia dividiendo entre `60.0` antes de entrenar?
2. ¿Qué forma tiene el tensor antes y después del modelo?
3. ¿Qué ventaja tiene una pérdida ponderada frente a MSE simple en campos de precipitación?
4. ¿Qué parte de la arquitectura permite mezclar información temporal y espacial?
5. ¿Por qué este entrenamiento corto sirve para explicar el pipeline, pero no para obtener un modelo científico final?

**Idea para clase:** abrir `earthformer_training_lab/architectures/earthformer_xy.py` y ubicar `EarthFormer_xy`, `earthformer_base`, `CuboidTransformerModel`, `CuboidTransformerEncoder` y `CuboidTransformerDecoder`.